In [ ]:
import os
import struct
from pathlib import Path

import numpy as np
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior()

# This notebook was originally TF1 + `tensorflow.examples` MNIST loader.
# TF2.x no longer ships `tensorflow.examples`, so we parse the MNIST IDX files
# that already exist in this repo under chap5_CNN/mnist/MNIST/raw.

learning_rate = 1e-4
keep_prob_rate = 0.7
max_epoch = 1000

def _read_idx(path):
    path = Path(path)
    with open(path, 'rb') as f:
        magic = struct.unpack('>I', f.read(4))[0]
        if magic == 2051:  # images
            n, rows, cols = struct.unpack('>III', f.read(12))
            data = np.frombuffer(f.read(), dtype=np.uint8)
            return data.reshape(n, rows, cols)
        if magic == 2049:  # labels
            n = struct.unpack('>I', f.read(4))[0]
            data = np.frombuffer(f.read(), dtype=np.uint8)
            return data.reshape(n)
        raise ValueError('unexpected IDX magic: %r' % (magic,))

def _find_mnist_raw_dir():
    candidates = [
        Path('mnist') / 'MNIST' / 'raw',
        Path('chap5_CNN') / 'mnist' / 'MNIST' / 'raw',
    ]
    for c in candidates:
        if (c / 'train-images-idx3-ubyte').exists() and (c / 'train-labels-idx1-ubyte').exists():
            return c
    raise FileNotFoundError('MNIST raw files not found under: %s' % candidates)

raw_dir = _find_mnist_raw_dir()
train_images = _read_idx(raw_dir / 'train-images-idx3-ubyte').astype(np.float32).reshape(-1, 784)
train_labels = _read_idx(raw_dir / 'train-labels-idx1-ubyte')
test_images = _read_idx(raw_dir / 't10k-images-idx3-ubyte').astype(np.float32).reshape(-1, 784)
test_labels = _read_idx(raw_dir / 't10k-labels-idx1-ubyte')

train_labels_oh = np.eye(10, dtype=np.float32)[train_labels]
test_labels_oh = np.eye(10, dtype=np.float32)[test_labels]

class _Split:
    def __init__(self, images, labels, shuffle=True):
        self.images = images
        self.labels = labels
        self._n = int(images.shape[0])
        self._shuffle = bool(shuffle)
        self._perm = np.arange(self._n)
        self._i = 0
        if self._shuffle:
            np.random.shuffle(self._perm)

    def next_batch(self, batch_size):
        batch_size = int(batch_size)
        if self._i + batch_size > self._n:
            self._i = 0
            if self._shuffle:
                np.random.shuffle(self._perm)
        idx = self._perm[self._i:self._i + batch_size]
        self._i += batch_size
        return self.images[idx], self.labels[idx]

class _MNIST:
    pass

mnist = _MNIST()
mnist.train = _Split(train_images, train_labels_oh, shuffle=True)
mnist.test = type('test', (), {'images': test_images, 'labels': test_labels_oh})()

def compute_accuracy(v_xs, v_ys):
    y_pre = sess.run(prediction, feed_dict={xs_ph: v_xs, keep_prob: 1})
    return float(np.mean(np.argmax(y_pre, axis=1) == np.argmax(v_ys, axis=1)))

def weight_variable(shape):
    initial = tf.truncated_normal(shape, stddev=0.1)
    return tf.Variable(initial)

def bias_variable(shape):
    initial = tf.constant(0.1, shape=shape)
    return tf.Variable(initial)

def conv2d(x, W):
    return tf.nn.conv2d(x, W, strides=[1, 1, 1, 1], padding='SAME')

def max_pool_2x2(x):
    return tf.nn.max_pool(x, ksize=[1, 2, 2, 1], strides=[1, 2, 2, 1], padding='SAME')

# define placeholder for inputs to network
xs_ph = tf.placeholder(tf.float32, [None, 784])
xs = xs_ph / 255.
ys = tf.placeholder(tf.float32, [None, 10])
keep_prob = tf.placeholder(tf.float32)
x_image = tf.reshape(xs, [-1, 28, 28, 1])

#  卷积层 1
W_conv1 = weight_variable([7, 7, 1, 32])
b_conv1 = bias_variable([32])
h_conv1 = tf.nn.relu(conv2d(x_image, W_conv1) + b_conv1)
h_pool1 = max_pool_2x2(h_conv1)

# 卷积层 2
W_conv2 = weight_variable([5, 5, 32, 64])
b_conv2 = bias_variable([64])
h_conv2 = tf.nn.relu(conv2d(h_pool1, W_conv2) + b_conv2)
h_pool2 = max_pool_2x2(h_conv2)

#  全连接层 1
W_fc1 = weight_variable([7 * 7 * 64, 1024])
b_fc1 = bias_variable([1024])
h_pool2_flat = tf.reshape(h_pool2, [-1, 7 * 7 * 64])
h_fc1 = tf.nn.relu(tf.matmul(h_pool2_flat, W_fc1) + b_fc1)
h_fc1_drop = tf.nn.dropout(h_fc1, keep_prob=keep_prob)

# 全连接层 2
W_fc2 = weight_variable([1024, 10])
b_fc2 = bias_variable([10])
prediction = tf.nn.softmax(tf.matmul(h_fc1_drop, W_fc2) + b_fc2)

# 交叉熵函数
cross_entropy = tf.reduce_mean(-tf.reduce_sum(ys * tf.log(prediction + 1e-8), reduction_indices=[1]))
train_step = tf.train.AdamOptimizer(learning_rate).minimize(cross_entropy)


import matplotlib.pyplot as plt

history_loss = []
history_acc = []
steps = []

with tf.Session() as sess:
    init = tf.global_variables_initializer()
    sess.run(init)

    for i in range(max_epoch):
        batch_xs, batch_ys = mnist.train.next_batch(100)
        _, loss_val = sess.run([train_step, cross_entropy], feed_dict={xs_ph: batch_xs, ys: batch_ys, keep_prob: keep_prob_rate})
        
        # 每10步采样一次指标，以便画出平滑的曲线
        if i % 10 == 0:
            acc = compute_accuracy(mnist.test.images[:1000], mnist.test.labels[:1000])
            history_loss.append(loss_val)
            history_acc.append(acc)
            steps.append(i)
            # 控制台每100步打印一次
            if i % 100 == 0:
                print(f"Step {i}, Loss: {loss_val:.4f}, Test Acc: {acc:.4f}")

# 绘制曲线
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(steps, history_loss, label='Train Loss', color='red')
plt.title('TF1 CNN: Loss')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(steps, history_acc, label='Test Accuracy', color='blue')
plt.title('TF1 CNN: Test Accuracy')
plt.xlabel('Steps')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.savefig('tf1_cnn_curve.png') # 保存总结图表
plt.show()

### 框架对比与实验结果总结 (TensorFlow 1.x)

1. **核心逻辑**：TF1.x 使用的是**静态计算图 (Static Computation Graph)** 的机制。在准备数据、搭建模型结构、定义网络层数 (`conv2d`, `max_pool`, `reshape`) 时，程序并没有真正计算任何数值，而是在提前搭建一张蓝图（并且大量使用 `tf.placeholder` 占位符作为数据管道）。这种方式虽然对初学者理解起来门槛较高、难以调试，但其底层的执行效率极高。
2. **训练执行**：只有在实例化并拉起 `with tf.Session() as sess:`，并且调用 `sess.run()` 同时用 `feed_dict` 喂入输入数据时，计算图才会开始工作，并且返回计算好的 Loss 和执行梯度更新。
3. **可视化表现**：从我们添加的代码和绘出的折线图来看，`tf.train.AdamOptimizer` 在 500 个 batch 步数内，Loss 会迅速下降并稳定，在测试集上的预测准确度 (`Test Accuracy`) 会在一开始大幅跃升到较高水平。这种基于底层的实现方法奠定了以后高阶 API (如 Keras) 的基础。